# 📈 Marketing Mix Model (MMM)
**Author:** Vincent Lepore | **Methodology:** Robyn-inspired — Adstock + Hill Saturation + OLS Decomposition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/marketing-analytics-portfolio/blob/main/notebooks/04_mmm_model.ipynb)

---

## What is MMM?
Marketing Mix Modeling answers: **"How much revenue did each channel actually drive — accounting for lag and diminishing returns?"**

Unlike MTA (which works at the individual user level), MMM works at the **aggregate level** using statistical regression. It's ideal for:
- Channels without user-level tracking (TV, OOH, Radio)
- Measuring long-term brand effects
- Budget optimization across channels

### The Three Transformations
```
Raw Spend → [Adstock] → Carry-over Effect → [Hill Saturation] → Effective Reach → [Regression] → Revenue
```

| Step | What it does | Why it matters |
|------|-------------|----------------|
| **Adstock** | Decays past spend into current period | Advertising has a memory effect |
| **Hill Saturation** | Applies diminishing returns curve | First $1K of TV has more impact than $100K |
| **OLS Regression** | Decomposes revenue into channel contributions | Tells you the $ value of each channel |

> **Used by:** Google's Meridian, Meta's Robyn, Nielsen, Analytic Partners, Ekimetrics

## 1. Setup & Data Loading

In [ ]:
!pip install numpy pandas scikit-learn scipy matplotlib -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import minimize
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')

CHANNEL_COLORS = {
    'google_brand_search':    '#00b4d8',
    'google_nonbrand_search': '#9b5de5',
    'google_remarketing':     '#00f5d4',
    'google_display_pmax':    '#fee440',
    'meta_prospecting':       '#f15bb5',
    'meta_retargeting':       '#06d6a0',
    'meta_other':             '#a8dadc',
}

df_gads = pd.read_csv("data/raw/google_ads_daily.csv")
df_meta = pd.read_csv("data/raw/meta_ads_daily.csv")
df_gads['date'] = pd.to_datetime(df_gads['segments.date'])
df_meta['date'] = pd.to_datetime(df_meta['date_start'])
print("✓ Data loaded")

## 2. Adstock Transformation — Modeling Carry-Over Effects

In [ ]:
def apply_adstock(spend_array, decay_rate):
    """
    Geometric Adstock Transformation
    
    Formula: adstock_t = spend_t + decay_rate * adstock_{t-1}
    
    This models the 'memory' of advertising:
    - A TV spot on Monday still influences purchases on Friday
    - Search ads have low decay (near-instant response)
    - Display/brand campaigns have high decay (longer brand echo)
    
    Args:
        spend_array: weekly spend values
        decay_rate:  0.0 = no carryover, 0.9 = very long memory
    
    Returns:
        adstocked spend array (same shape)
    """
    adstocked = np.zeros(len(spend_array))
    adstocked[0] = spend_array[0]
    for t in range(1, len(spend_array)):
        adstocked[t] = spend_array[t] + decay_rate * adstocked[t-1]
    return adstocked

# Visual demonstration
np.random.seed(42)
demo_weeks = 20
demo_spend = np.zeros(demo_weeks)
demo_spend[0] = 10000   # One-time spend burst in week 1

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor('#0d1117')

for ax, decay, color in zip(axes, [0.2, 0.5, 0.8], ['#00b4d8','#00f5d4','#f15bb5']):
    ads = apply_adstock(demo_spend, decay)
    ax.bar(range(demo_weeks), ads, color=color, alpha=0.8)
    ax.axvline(0, color='white', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.set_title(f'Decay Rate = {decay}', fontweight='bold')
    ax.set_xlabel('Weeks after campaign'); ax.set_ylabel('Effective Impressions')
    ax.set_facecolor('#161b22'); ax.grid(alpha=0.3, color='white')
    half_life = int(np.log(0.5) / np.log(decay)) if decay > 0 else 0
    ax.text(0.6, 0.8, f'Half-life: ~{half_life}w', transform=ax.transAxes, 
            color=color, fontsize=10, fontweight='bold')

plt.suptitle('Adstock Decay — How Advertising Lingers Over Time', fontsize=13, fontweight='bold', color='white')
plt.tight_layout(); plt.show()
print("Key insight: High decay rate = brand/awareness channels. Low decay = direct response channels.")

## 3. Hill Saturation — Modeling Diminishing Returns

In [ ]:
def apply_hill(x, alpha=2.0, gamma=0.5):
    """
    Hill (S-curve) Saturation Function
    
    Formula: y = x^alpha / (x^alpha + gamma^alpha)
    
    - alpha: steepness of the S-curve (higher = more abrupt saturation)  
    - gamma: inflection point as fraction of max spend (where returns start diminishing)
    
    Economic intuition: 
    - First dollar of media spend reaches the most receptive audience
    - Each additional dollar reaches progressively less receptive people
    - Eventually you're just paying to show ads to people who will never buy
    """
    x_scaled = x / x.max() if x.max() > 0 else x
    return x_scaled**alpha / (x_scaled**alpha + gamma**alpha)

x_range = np.linspace(0, 1, 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

# Effect of alpha
ax = axes[0]; ax.set_facecolor('#161b22')
for alpha, color in [(0.5,'#f15bb5'), (1.0,'#fee440'), (2.0,'#00b4d8'), (3.0,'#00f5d4')]:
    y = x_range**alpha / (x_range**alpha + 0.5**alpha)
    ax.plot(x_range, y, label=f'α={alpha}', color=color, linewidth=2)
ax.set_title('Effect of Alpha (Steepness)', fontweight='bold')
ax.set_xlabel('Spend (normalized)'); ax.set_ylabel('Response Rate')
ax.legend(); ax.grid(alpha=0.3, color='white')

# Effect of gamma
ax = axes[1]; ax.set_facecolor('#161b22')
for gamma, color in [(0.3,'#f15bb5'), (0.5,'#fee440'), (0.7,'#00b4d8'), (0.9,'#00f5d4')]:
    y = x_range**2 / (x_range**2 + gamma**2)
    ax.plot(x_range, y, label=f'γ={gamma}', color=color, linewidth=2)
ax.set_title('Effect of Gamma (Inflection Point)', fontweight='bold')
ax.set_xlabel('Spend (normalized)'); ax.set_ylabel('Response Rate')
ax.legend(); ax.grid(alpha=0.3, color='white')

plt.suptitle('Hill Saturation — Modeling Diminishing Returns on Ad Spend', fontsize=13, fontweight='bold', color='white')
plt.tight_layout(); plt.show()

## 4. Aggregate Spend to Weekly & Fit MMM

In [ ]:
# Aggregate to weekly
def get_channel(name, source):
    n = str(name)
    if source == 'gads':
        if 'Brand' in n: return 'google_brand_search'
        if any(w in n for w in ['Generic','Competitor','Category']): return 'google_nonbrand_search'
        if any(w in n for w in ['Remarket','Retarget']): return 'google_remarketing'
        return 'google_display_pmax'
    else:
        if any(w in n for w in ['Prospect','Aware']): return 'meta_prospecting'
        if any(w in n for w in ['Retarget','LAL']): return 'meta_retargeting'
        return 'meta_other'

df_gads['mmm_channel'] = df_gads['campaign.name'].apply(lambda x: get_channel(x, 'gads'))
df_meta['mmm_channel'] = df_meta['campaign_name'].apply(lambda x: get_channel(x, 'meta'))

gads_wk = df_gads.groupby([pd.Grouper(key='date',freq='W-MON'),'mmm_channel'])['metrics.cost'].sum().reset_index()
meta_wk = df_meta.groupby([pd.Grouper(key='date',freq='W-MON'),'mmm_channel'])['spend'].sum().reset_index()
gads_wk.columns = ['week','channel','spend']
meta_wk.columns = ['week','channel','spend']

spend_wide = pd.concat([gads_wk,meta_wk]).pivot_table(index='week',columns='channel',values='spend',aggfunc='sum').fillna(0).reset_index()
CHANNELS = [c for c in spend_wide.columns if c != 'week']
n_weeks  = len(spend_wide)

# Simulated revenue (in production: join NetSuite weekly revenue)
np.random.seed(42)
t = np.arange(n_weeks)
TRUE_PARAMS = {
    'google_brand_search':   {'adstock':0.25,'alpha':2.5,'gamma':0.4,'roi':3.8},
    'google_nonbrand_search':{'adstock':0.35,'alpha':2.0,'gamma':0.5,'roi':2.9},
    'google_remarketing':    {'adstock':0.40,'alpha':1.8,'gamma':0.45,'roi':3.2},
    'google_display_pmax':   {'adstock':0.55,'alpha':1.5,'gamma':0.6,'roi':1.8},
    'meta_prospecting':      {'adstock':0.60,'alpha':1.4,'gamma':0.65,'roi':1.6},
    'meta_retargeting':      {'adstock':0.45,'alpha':2.0,'gamma':0.50,'roi':2.7},
    'meta_other':            {'adstock':0.50,'alpha':1.3,'gamma':0.60,'roi':1.5},
}

base_rev = 75000 * (1 + 0.003*t) + 12000*np.sin(2*np.pi*t/52)
channel_rev = {}
for ch in CHANNELS:
    if ch in TRUE_PARAMS and ch in spend_wide.columns:
        p = TRUE_PARAMS[ch]
        ads = apply_adstock(spend_wide[ch].values, p['adstock'])
        sat = apply_hill(ads, p['alpha'], p['gamma'])
        channel_rev[ch] = sat * p['roi'] * spend_wide[ch].mean()

revenue = base_rev + sum(channel_rev.values()) + np.random.normal(0, 4000, n_weeks)
spend_wide['revenue']  = revenue
spend_wide['week_num'] = t
spend_wide['sin_52']   = np.sin(2*np.pi*t/52)
spend_wide['cos_52']   = np.cos(2*np.pi*t/52)

# Grid search for best adstock + saturation params
best_params = {}
for ch in CHANNELS:
    best_r2, best_p = -999, {'adstock':0.4,'alpha':2.0,'gamma':0.5}
    for decay in [0.2,0.3,0.4,0.5,0.6,0.7]:
        for alpha in [1.0,1.5,2.0,2.5]:
            for gamma in [0.3,0.4,0.5,0.6,0.7]:
                if ch in spend_wide.columns:
                    ads = apply_adstock(spend_wide[ch].values, decay)
                    sat = apply_hill(ads, alpha, gamma)
                    if sat.std() > 0:
                        corr = np.corrcoef(sat, revenue)[0,1]
                        if corr > best_r2:
                            best_r2, best_p = corr, {'adstock':decay,'alpha':alpha,'gamma':gamma}
    best_params[ch] = best_p

# Build feature matrix
X_dict = {'trend': t, 'sin_52': spend_wide['sin_52'].values, 'cos_52': spend_wide['cos_52'].values}
for ch in CHANNELS:
    if ch in spend_wide.columns:
        p = best_params[ch]
        ads = apply_adstock(spend_wide[ch].values, p['adstock'])
        X_dict[f'{ch}_t'] = apply_hill(ads, p['alpha'], p['gamma'])

X = pd.DataFrame(X_dict)
y = revenue

model = LinearRegression().fit(X, y)
y_pred = model.predict(X)
print(f"Model R²:   {r2_score(y, y_pred):.4f}")
print(f"Model MAPE: {mean_absolute_percentage_error(y, y_pred):.2%}")

## 5. Revenue Decomposition & Visualization

In [ ]:
coefs = dict(zip(X.columns, model.coef_))
intercept = model.intercept_

contributions = {}
for ch in CHANNELS:
    col  = f'{ch}_t'
    coef = max(0, coefs.get(col, 0))
    contributions[ch] = coef * X.get(col, pd.Series(np.zeros(n_weeks))).values if col in X else np.zeros(n_weeks)

base = intercept + coefs.get('trend',0)*t
seasonal = coefs.get('sin_52',0)*X['sin_52'].values + coefs.get('cos_52',0)*X['cos_52'].values
total_rev = revenue.sum()

print("Revenue Decomposition:")
print(f"  {'Component':<30} {'$':>12}  {'%':>6}")
print("-"*55)
print(f"  {'Base / Organic':<30} ${base.sum():>10,.0f}  {base.sum()/total_rev:>5.1%}")
print(f"  {'Seasonality':<30} ${seasonal.sum():>10,.0f}  {seasonal.sum()/total_rev:>5.1%}")
for ch, contrib in sorted(contributions.items(), key=lambda x: -x[1].sum()):
    pct = contrib.sum()/total_rev
    if pct > 0.001:
        print(f"  {ch:<30} ${contrib.sum():>10,.0f}  {pct:>5.1%}")

# Stacked area chart
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#0d1117')
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)
ax1 = fig.add_subplot(gs[0, :])

# Stacked area
stack_data  = [base + seasonal] + [contributions[ch] for ch in CHANNELS if contributions[ch].sum() > 0]
stack_labels= ['Base + Seasonal'] + [ch.replace('_',' ').title() for ch in CHANNELS if contributions[ch].sum() > 0]
stack_colors= ['#374151'] + [CHANNEL_COLORS.get(ch,'#adb5bd') for ch in CHANNELS if contributions[ch].sum() > 0]

ax1.stackplot(range(n_weeks), stack_data, labels=stack_labels, colors=stack_colors, alpha=0.85)
ax1.plot(range(n_weeks), revenue, color='white', linewidth=1.5, linestyle='--', label='Actual Revenue', alpha=0.8)
ax1.set_facecolor('#161b22')
ax1.legend(fontsize=8, loc='upper left', ncol=3)
ax1.set_title('Weekly Revenue Decomposition — Actual vs. Modeled', fontsize=13, fontweight='bold', color='white')
ax1.set_ylabel('Revenue ($)'); ax1.grid(alpha=0.2, color='white')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))

# ROAS bar
ax2 = fig.add_subplot(gs[1, 0]); ax2.set_facecolor('#161b22')
roas_data = {ch: (contributions[ch].sum() / spend_wide.get(ch, pd.Series([1])).sum()) 
             for ch in CHANNELS if contributions[ch].sum() > 0}
roas_sorted = sorted(roas_data.items(), key=lambda x: -x[1])
ch_names, roas_vals = zip(*roas_sorted)
colors = [CHANNEL_COLORS.get(ch,'#adb5bd') for ch in ch_names]
bars = ax2.barh([c.replace('_',' ').title() for c in ch_names], roas_vals, color=colors, alpha=0.85)
ax2.axvline(3, color='white', linestyle='--', linewidth=1, alpha=0.5, label='3x benchmark')
ax2.set_xlabel('ROAS'); ax2.set_title('Channel ROAS (MMM)', fontweight='bold'); ax2.grid(axis='x', alpha=0.3, color='white')
ax2.legend(fontsize=9)
for bar, val in zip(bars, roas_vals):
    ax2.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2, f'{val:.2f}x', va='center', fontsize=9)

# Adstock params
ax3 = fig.add_subplot(gs[1, 1]); ax3.set_facecolor('#161b22')
ch_labels = [c.replace('_',' ').title() for c in CHANNELS]
decay_vals = [best_params[ch]['adstock'] for ch in CHANNELS]
ax3.barh(ch_labels, decay_vals, color=[CHANNEL_COLORS.get(ch,'#adb5bd') for ch in CHANNELS], alpha=0.85)
ax3.set_xlabel('Adstock Decay Rate'); ax3.set_title('Fitted Adstock Parameters', fontweight='bold')
ax3.grid(axis='x', alpha=0.3, color='white')
ax3.axvline(0.5, color='#fee440', linestyle='--', linewidth=1, alpha=0.7, label='Mid-point')
ax3.legend(fontsize=9)

plt.savefig('data/processed/mmm_decomp.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"\n✅ MMM complete — R²: {r2_score(y,y_pred):.4f}")